## Loading the dataset

### imports

In [18]:
from datasets import load_dataset
import pandas as pd
import re


### download WikiText-2

In [5]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

print(dataset)

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


### Inspecting

In [8]:
for split_name in dataset.keys():
    print(split_name, "examples:", len(dataset[split_name]))

test examples: 4358
train examples: 36718
validation examples: 3760


In [15]:
for i in range(30):
    print(repr(dataset["train"][i]["text"]))

''
' = Valkyria Chronicles III = \n'
''
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
" The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving 

In [ ]:
def inspect_split(split):
    texts = [example["text"] for example in split]
    
    return {
        "total_lines": len(texts),
        "empty_lines": sum(not text.strip() for text in texts),
        "non_empty_lines": sum(bool(text.strip()) for text in texts),
        "total_characters": sum(len(text) for text in texts),
    }
    
original_stats = pd.DataFrame({name: inspect_split(dataset[name]) for name in dataset.keys()}).T

original_stats

,total_lines,empty_lines,non_empty_lines,total_characters
test,4358,1467,2891,1285622
train,36718,12951,23767,10892990
validation,3760,1299,2461,1142150


In [20]:
def inspect_artifacts(split):
    texts = [example["text"] for example in split]

    return {
        "heading_lines": sum(bool(re.fullmatch(r"=+.*=+", text.strip())) for text in texts if text.strip()),
        "lines_with_at_dash_at": sum("@-@" in text for text in texts),
        "lines_with_at_comma_at": sum("@,@" in text for text in texts),
        "lines_with_at_dot_at": sum("@.@" in text for text in texts),
    }

artifact_stats = pd.DataFrame({split_name: inspect_artifacts(dataset[split_name])for split_name in dataset.keys()}).T

artifact_stats

,heading_lines,lines_with_at_dash_at,lines_with_at_comma_at,lines_with_at_dot_at
test,706,1023,168,204
train,6211,8198,1331,1346
validation,620,891,200,117


In [21]:
for i, example in enumerate(dataset["train"]):
    text = example["text"]

    if text.strip() and (re.fullmatch(r"=+.*=+", text.strip()) or "@-@" in text or "@,@" in text or "@.@" in text):
        print(repr(text))

    if i > 500:
        break

' = Valkyria Chronicles III = \n'
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
' = = Gameplay = = \n'
" As with previous Valkyira Chronicles games , Valkyria Chronicles III is a tactical role @-@ playing game where players take control of a military unit and take part in missions against enemy forces . Stories are told through comic 

### Cleaning WikiText formatting artifacts

In [ ]:
def clean_line(text):
    text = text.strip()
    if not text: 
        return None
    if re.fullmatch(r"=+.*=+", text): # skip heading-only
        return None 
    
    # then we want to restore 
    text = text.replace("@-@", "-")
    text = text.replace("@,@", ",")
    text = text.replace("@.@", ".")
    
    # to replace repeated whitespace with a single space
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None


examples = [" = = Gameplay = = ", "role @-@ playing video game", "102 @,@ 779 units", "1 @.@ 38 goals against average","   ",]

for example in examples:
    print("Before:", repr(example))
    print("After: ", repr(clean_line(example)))
    print()

Before: ' = = Gameplay = = '
After:  None

Before: 'role @-@ playing video game'
After:  'role - playing video game'

Before: '102 @,@ 779 units'
After:  '102 , 779 units'

Before: '1 @.@ 38 goals against average'
After:  '1 . 38 goals against average'

Before: '   '
After:  None

